# EDA - Exploração das Tabelas

Este notebook resume as primeiras análises do desafio utilizando as tabelas de voos, companhias aéreas e aeroportos. Aqui são apresentados diagnósticos estatísticos, mapeamentos de campos e visualizações.

## Visão Geral das Tabelas
- airlines: Companhias aéreas
- airports: Aeroportos, localização
- flights: Voos, atrasos, cancelamentos, rotas

In [0]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Carregar airlines e airports
airlines_df = spark.read.table('workspace.postech_tech_challenge_f3.airlines')
airports_df = spark.read.table('workspace.postech_tech_challenge_f3.airports')
# Carregar linhas de flights para EDA inicial
flights_df = spark.read.table('workspace.postech_tech_challenge_f3.flights')

In [0]:
# Estatísticas das companhias aéreas

num_airlines = airlines_df.count()
dist_airlines = airlines_df.select('AIRLINE').toPandas()['AIRLINE'].value_counts()
dist_airlines.head()

In [0]:
# Estatísticas dos aeroportos

num_airports = airports_df.count()
city_airports = airports_df.groupBy('CITY').count().orderBy('count', ascending=False).limit(10)
city_airports_pd = city_airports.toPandas()
city_airports_pd.head()

In [0]:
# Estatísticas dos voos: atrasos, cancelamentos e principais rotas

import matplotlib.pyplot as plt

flight_stats = flights_df.groupBy('AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT')\
    .agg({'DEPARTURE_DELAY': 'avg', 'CANCELLED': 'sum', 'ORIGIN_AIRPORT': 'count'})\
    .orderBy('count(ORIGIN_AIRPORT)', ascending=False)\
    .limit(10)
flight_stats_pd = flight_stats.toPandas()

# Visualização: Bar chart das rotas com mais voos
plt.figure(figsize=(12, 5))
plt.bar(range(len(flight_stats_pd)), flight_stats_pd['count(ORIGIN_AIRPORT)'])
plt.xticks(range(len(flight_stats_pd)), [f"{r['ORIGIN_AIRPORT']}-{r['DESTINATION_AIRPORT']}" for i, r in flight_stats_pd.iterrows()], rotation=45)
plt.title('Top 10 rotas por volume de voos')
plt.ylabel('Número de voos')
plt.tight_layout()
plt.show()

In [0]:
# Diagnóstico de atraso e cancelamento das principais rotas

plt.figure(figsize=(12, 5))
plt.bar(range(len(flight_stats_pd)), flight_stats_pd['avg(DEPARTURE_DELAY)'])
plt.xticks(range(len(flight_stats_pd)), [f"{r['ORIGIN_AIRPORT']}-{r['DESTINATION_AIRPORT']}" for i, r in flight_stats_pd.iterrows()], rotation=45)
plt.title('Atraso médio nas top 10 rotas')
plt.ylabel('Atraso médio (minutos)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.bar(range(len(flight_stats_pd)), flight_stats_pd['sum(CANCELLED)'])
plt.xticks(range(len(flight_stats_pd)), [f"{r['ORIGIN_AIRPORT']}-{r['DESTINATION_AIRPORT']}" for i, r in flight_stats_pd.iterrows()], rotation=45)
plt.title('Cancelamentos nas top 10 rotas')
plt.ylabel('Total de cancelamentos')
plt.tight_layout()
plt.show()

In [0]:
# Estatísticas descritivas dos voos

# Estatística de atraso e cancelamento

# Resumo de delay
summary_delay = flights_df.select('DEPARTURE_DELAY').summary().toPandas()
display(summary_delay)

# Resumo de voos cancelados
cancel_stats = flights_df.groupBy('CANCELLED').count().toPandas()
display(cancel_stats)

# Resumo de desvio
if 'DIVERTED' in flights_df.columns:
    divert_stats = flights_df.groupBy('DIVERTED').count().toPandas()
    display(divert_stats)

# Estatística por dia da semana
weekday_stats = flights_df.groupBy('DAY_OF_WEEK').agg({'DEPARTURE_DELAY':'avg', 'CANCELLED':'sum'}).orderBy('DAY_OF_WEEK').toPandas()
display(weekday_stats)

# Estatística por mês
month_stats = flights_df.groupBy('MONTH').agg({'DEPARTURE_DELAY':'avg', 'CANCELLED':'sum'}).orderBy('MONTH').toPandas()
display(month_stats)

In [0]:
# Visualizações de insights sobre delay, cancelamento e sazonalidade

import seaborn as sns
import matplotlib.pyplot as plt

# Boxplot atrasos por companhia aérea
plt.figure(figsize=(12,6))
sns.boxplot(x='AIRLINE', y='DEPARTURE_DELAY', data=flights_df.toPandas())
plt.title('Distribuição de atrasos por companhia aérea')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Histograma dos atrasos
plt.figure(figsize=(10,5))
flights_pd = flights_df.select('DEPARTURE_DELAY').toPandas()
plt.hist(flights_pd['DEPARTURE_DELAY'].dropna(), bins=40)
plt.title('Histograma dos atrasos de partida')
plt.xlabel('Minutos de atraso')
plt.ylabel('Frequência')
plt.tight_layout()
plt.show()

# Cancelamentos por mês
month_stats = flights_df.groupBy('MONTH').agg({'CANCELLED':'sum'}).orderBy('MONTH').toPandas()
plt.figure(figsize=(8,4))
plt.bar(month_stats['MONTH'], month_stats['sum(CANCELLED)'])
plt.title('Cancelamentos por mês')
plt.xlabel('Mês')
plt.ylabel('Total de cancelamentos')
plt.tight_layout()
plt.show()

In [0]:
# Análise de correlação entre variáveis numéricas

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Selecionar variáveis numéricas relevantes para análise de correlação
numeric_cols = ['MONTH', 'DAY', 'DAY_OF_WEEK', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 
                'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'SCHEDULED_TIME', 'ELAPSED_TIME', 
                'AIR_TIME', 'DISTANCE', 'CANCELLED', 'DIVERTED']

# Filtrar apenas colunas que existem no DataFrame
available_cols = [col for col in numeric_cols if col in flights_df.columns]

# Converter para pandas e calcular correlação
flights_sample = flights_df.select(available_cols).sample(False, 0.1, seed=42).toPandas()
corr_matrix = flights_sample.corr()

# Visualizar matriz de correlação com heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlação - Variáveis do Dataset de Voos', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# Exibir as correlações mais fortes com DEPARTURE_DELAY
if 'DEPARTURE_DELAY' in corr_matrix.columns:
    print("\n=== Correlações mais fortes com DEPARTURE_DELAY ===")
    delay_corr = corr_matrix['DEPARTURE_DELAY'].sort_values(ascending=False)
    display(delay_corr)